In [1]:
import pandas as pd
import ast
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def convert(text):
    try:
        return [i['name'] for i in ast.literal_eval(text)]
    except:
        return []

In [3]:
def preprocess():
    movies = pd.read_csv(r"C:\Users\Lonovo\OneDrive\Documents\Movie Recommendation System\movie_dataset.csv")

    # Handle missing values
    movies.dropna(subset=['genres', 'director'], inplace=True)

    # Convert genres
    movies['genres'] = movies['genres'].apply(convert)

    # Convert director
    movies['director'] = movies['director'].apply(lambda x: [str(x)])

    # Add language feature
    movies['original_language'] = movies['original_language'].apply(lambda x: [x])

    # Create tags
    movies['tags'] = movies['genres'] + movies['director'] + movies['original_language']

    # Clean tags
    movies['tags'] = movies['tags'].apply(
        lambda x: [str(i).lower().replace(" ", "") for i in x]
    )

    # Convert list → string
    movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

    # Final dataframe
    df = movies[['id', 'title', 'tags']].copy()
    df = df.rename(columns={'id': 'movie_id'})

    # Remove duplicates
    df.drop_duplicates(subset='title', inplace=True)

    return df

In [4]:
df = preprocess()
print(df.head())

   movie_id                                     title                 tags
0     19995                                    Avatar      jamescameron en
1       285  Pirates of the Caribbean: At World's End     goreverbinski en
2    206647                                   Spectre         sammendes en
3     49026                     The Dark Knight Rises  christophernolan en
4     49529                               John Carter     andrewstanton en


In [5]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(df['tags']).toarray()

In [6]:
similarity = cosine_similarity(vectors)

In [7]:
def recommend(movie):
    movie = movie.lower()

    if movie not in df['title'].str.lower().values:
        print("❌ Movie not found!")
        return

    index = df[df['title'].str.lower() == movie].index[0]
    distances = similarity[index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print(f"\n🎬 Recommendations for {movie.title()}:\n")
    for i in movies_list:
        print(df.iloc[i[0]].title)

In [8]:
recommend("Avatar")


🎬 Recommendations for Avatar:

Titanic
Terminator 2: Judgment Day
True Lies
The Abyss
Aliens


In [9]:
pickle.dump(df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))